In [25]:
# Módulos estándar
import gc
import shutil
from datetime import datetime, timedelta
from pathlib import Path

# Análisis de datos y procesamiento geoespacial
import numpy as np
import pandas as pd
import xarray as xr
from pyproj import Proj
from tqdm import tqdm
from goes2go import GOES
from datetime import datetime
import time

In [29]:
# ==============================================================================
# 1. PARÁMETROS DEL PIPELINE
# ==============================================================================
# Periodo de procesamiento
START_DATE = "2026-01-01"
END_DATE   = "2026-01-31"

# Coordenadas de la Unidad de Astronomía (OAQ)
LAT_OAQ = -0.2150
LON_OAQ = -78.5025

# Ventana espacial para el cálculo de estadísticas (píxeles)
WINDOW_SIZE = 5

# Producto GOES a procesar
PRODUCT = "ABI-L2-MCMIPF"
SATELLITE = "G18"

# ==============================================================================
# 2. DEFINICIÓN DE RUTAS (PROJECT ROOT & DATA DIR)
# ==============================================================================
PROJECT_ROOT = Path(
    r"C:\Users\Brian OAQ\Desktop\OAQ\2026\12. Manuscrito de un artículo Modelo predictivo de nubosidad para observaciones astronómicas\OAQ-AstroForecast"
)

# CORRECCIÓN: Apuntamos directamente a donde tu script de descarga guardó los NetCDF
RAW_GOES_DIR = Path(r"C:\Users\Brian OAQ\data\noaa-goes18\ABI-L2-MCMIPF")
TEMP_GOES_DIR = RAW_GOES_DIR / "temp" # Por si la descarga usa una subcarpeta temporal

# Datos Procesados (Parquet) - Estos sí se quedan firmes en la raíz de tu proyecto
PROCESSED_GOES_DIR = PROJECT_ROOT / "data" / "processed" / "goes"
DAILY_DIR          = PROCESSED_GOES_DIR / "daily"
LOGS_DIR           = PROCESSED_GOES_DIR / "logs"

# Archivos de salida y logs
FINAL_PARQUET    = PROCESSED_GOES_DIR / "goes_oaq_dataset.parquet"
PROCESSING_LOG   = LOGS_DIR / "processing_log.csv"
FAILED_FILES_LOG = LOGS_DIR / "failed_files.csv"

# ==============================================================================
# 3. CREACIÓN Y VERIFICACIÓN DE DIRECTORIOS
# ==============================================================================
if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f"El directorio raíz del proyecto no existe: {PROJECT_ROOT}")

DIRECTORIES = [
    RAW_GOES_DIR,
    TEMP_GOES_DIR,
    PROCESSED_GOES_DIR,
    DAILY_DIR,
    LOGS_DIR
]

for directory in DIRECTORIES:
    directory.mkdir(parents=True, exist_ok=True)

print(" Base del proyecto detectada correctamente.")
print(f" Directorios de datos y logs verificados en: {PROCESSED_GOES_DIR.relative_to(PROJECT_ROOT)}")

 Base del proyecto detectada correctamente.
 Directorios de datos y logs verificados en: data\processed\goes


In [3]:
# ==============================================================================
# 4. GESTIÓN DE LOGS Y LOGÍSTICA DE ARCHIVOS
# ==============================================================================

def initialize_logs():
    """
    Crea los archivos de log con sus encabezados si no existen en el disco.
    """
    if not PROCESSING_LOG.exists():
        with open(PROCESSING_LOG, mode="w", encoding="utf-8") as f:
            f.write("date,files_found,files_processed,files_failed,status\n")

    if not FAILED_FILES_LOG.exists():
        with open(FAILED_FILES_LOG, mode="w", encoding="utf-8") as f:
            f.write("date,filename,error\n")

    print("✓ Logs inicializados y validados de forma segura.")


def log_processing_day(date, files_found, files_processed, files_failed, status):
    """
    Registra de manera eficiente el resultado de procesamiento de un día.
    """
    with open(PROCESSING_LOG, mode="a", encoding="utf-8") as f:
        f.write(f"{date},{files_found},{files_processed},{files_failed},{status}\n")


def log_failed_file(date, filename, error_message):
    """
    Registra un archivo NetCDF específico que resultó corrupto o fallido.
    """
    # Limpiamos comas o saltos de línea del error para evitar romper el formato CSV
    clean_error = str(error_message).replace(",", ";").replace("\n", " ")
    with open(FAILED_FILES_LOG, mode="a", encoding="utf-8") as f:
        f.write(f"{date},{Path(filename).name},{clean_error}\n")


def day_already_processed(date):
    """
    Verifica la existencia del archivo parquet diario para evitar reprocesamientos.
    """
    daily_file = DAILY_DIR / f"{date}.parquet"
    return daily_file.exists()

# Inicialización automática al ejecutar la celda
initialize_logs()

✓ Logs inicializados y validados de forma segura.


In [8]:
# ==============================================================================
# 5. DIAGNÓSTICO E INSPECCIÓN DE NETCDF DISPONIBLES
# ==============================================================================

# Cambiado a RAW_GOES_DIR para alinearse con la estructura de directorios del proyecto
print(f"Buscando archivos crudos en: {RAW_GOES_DIR}")

# Búsqueda recursiva de archivos NetCDF del GOES
nc_files = list(RAW_GOES_DIR.rglob("*.nc"))
total_files = len(nc_files)

print(f"✓ Total de archivos .nc encontrados en el repositorio: {total_files}")

if total_files > 0:
    print("\n--- Muestra de los primeros 5 archivos detectados ---")
    for file_path in nc_files[:5]:
        print(f"  • {file_path.relative_to(PROJECT_ROOT) if PROJECT_ROOT in file_path.parents else file_path.name}")
    
    # Declaración del archivo de prueba para las siguientes fases
    sample_file = nc_files[0]
    print(f"\n✓ Archivo asignado para pruebas unitarias: {sample_file.name}")
else:
    sample_file = None
    print("\n Alerta: No se encontraron archivos NetCDF en el directorio especificado.")

Buscando archivos crudos en: C:\Users\Brian OAQ\data\noaa-goes18\ABI-L2-MCMIPF
✓ Total de archivos .nc encontrados en el repositorio: 551

--- Muestra de los primeros 5 archivos detectados ---
  • OR_ABI-L2-MCMIPF-M6_G18_s20260240000209_e20260240009530_c20260240009586.nc
  • OR_ABI-L2-MCMIPF-M6_G18_s20260240010209_e20260240019529_c20260240019587.nc
  • OR_ABI-L2-MCMIPF-M6_G18_s20260240020209_e20260240029529_c20260240029585.nc
  • OR_ABI-L2-MCMIPF-M6_G18_s20260240030209_e20260240039530_c20260240039587.nc
  • OR_ABI-L2-MCMIPF-M6_G18_s20260240040209_e20260240049518_c20260240049592.nc

✓ Archivo asignado para pruebas unitarias: OR_ABI-L2-MCMIPF-M6_G18_s20260240000209_e20260240009530_c20260240009586.nc


In [10]:
# ==============================================================================
# 6. CARGA E INSPECCIÓN DE METADATOS DEL ARCHIVO GOES
# ==============================================================================

if sample_file and sample_file.exists():
    print(f"Abriendo de forma segura: {sample_file.name}\n")
    
    # Eliminamos chunks y engine para evitar la dependencia de dask y abrirlo directamente
    with xr.open_dataset(sample_file) as ds:
        
        # 1. Extracción del tiempo central de la observación (t)
        # Convertimos a timestamp nativo de pandas para que sea legible
        goes_time = pd.to_datetime(ds.t.values)
        print(f" Fecha/Hora central del archivo (UTC): {goes_time}")
        print(f" Satélite original: {ds.attrs.get('platform_ID', 'No definido')}")
        print(f" Producto detectado: {ds.attrs.get('product_name', 'No definido')}")
        print("-" * 75)
        
        # 2. Listado resumido de variables disponibles (Bandas Reflectancia/Emisión y Geometría)
        variables_list = list(ds.variables)
        print(f"✓ Total de variables en el dataset: {len(variables_list)}")
        
        # Clasificamos las variables clave para el reporte visual
        bandas_cmip = [v for v in variables_list if v.startswith("CMIP")]
        print(f"  • Canales de reflectancia/brillo detectados ({len(bandas_cmip)}): {bandas_cmip}")
        print(f"  • Variables de proyección espacial: {[v for v in variables_list if 'projection' in v.lower() or v in ['x', 'y']]}")
        
        print("-" * 75)
        # 3. Vista resumida de la estructura interna del dataset de xarray
        print(ds)
        
else:
    print("⚠️ No hay un archivo de prueba válido asignado. Revisa la celda anterior.")

Abriendo de forma segura: OR_ABI-L2-MCMIPF-M6_G18_s20260240000209_e20260240009530_c20260240009586.nc

 Fecha/Hora central del archivo (UTC): 2026-01-24 00:05:07.031083008
 Satélite original: G18
 Producto detectado: No definido
---------------------------------------------------------------------------
✓ Total de variables en el dataset: 161
  • Canales de reflectancia/brillo detectados (0): []
  • Variables de proyección espacial: ['y', 'x', 'goes_imager_projection']
---------------------------------------------------------------------------
<xarray.Dataset> Size: 4GB
Dimensions:                                 (y: 5424, x: 5424,
                                             number_of_time_bounds: 2,
                                             number_of_image_bounds: 2, band: 1)
Coordinates: (12/37)
    t                                       datetime64[ns] 8B 2026-01-24T00:0...
  * y                                       (y) float32 22kB 0.1518 ... -0.1518
  * x                      

In [11]:
# ==============================================================================
# 7. CODIFICACIÓN DEL MOTOR DE EXTRACCIÓN ESPACIAL (OAQ CORE)
# ==============================================================================

def get_oaq_pixel(sample_file, lon_oaq=LON_OAQ, lat_oaq=LAT_OAQ):
    """
    Calcula mediante proyección geoestacionaria los índices (ix, iy) fijos 
    del píxel más cercano a las coordenadas geográficas del OAQ.
    """
    with xr.open_dataset(sample_file) as ds:
        # Extraer parámetros geométricos integrados en el NetCDF
        sat_h = ds.goes_imager_projection.perspective_point_height.item()
        sat_lon = ds.goes_imager_projection.longitude_of_projection_origin.item()
        sat_sweep = ds.goes_imager_projection.sweep_angle_axis
        
        # Configurar la proyección geoestacionaria inversa
        p = Proj(proj="geos", h=sat_h, lon_0=sat_lon, sweep=sat_sweep)
        x_oaq, y_oaq = p(lon_oaq, lat_oaq)
        
        # Convertir a radianes de escaneo satelital
        x_scan = x_oaq / sat_h
        y_scan = y_oaq / sat_h
        
        # Encontrar los índices más cercanos vectorizadamente en xarray
        ix = int(np.abs(ds.x - x_scan).argmin().item())
        iy = int(np.abs(ds.y - y_scan).argmin().item())
        
    return ix, iy


def extract_oaq_features(ds, ix, iy, window=WINDOW_SIZE):
    """
    Extrae las estadísticas de reflectancia/brillo para los canales C13, C15 y C16
    en la ventana espacial definida, computando las diferencias BTD para nubes.
    """
    if window % 2 == 0:
        raise ValueError("La ventana espacial (window) debe ser un número impar.")
        
    half = window // 2
    channels = ["CMI_C13", "CMI_C15", "CMI_C16"]
    features = {}
    
    # 1. Extracción estadística por canal
    for ch in channels:
        subset = ds[ch].isel(
            y=slice(iy - half, iy + half + 1),
            x=slice(ix - half, ix + half + 1)
        )
        
        prefix = ch.lower()
        features[f"{prefix}_mean"] = float(subset.mean().item())
        features[f"{prefix}_std"] = float(subset.std().item())
        features[f"{prefix}_min"] = float(subset.min().item())
        features[f"{prefix}_max"] = float(subset.max().item())
        
    # 2. Cálculo de Diferencias de Temperatura de Brillo (BTD)
    features["btd_13_15"] = features["cmi_c13_mean"] - features["cmi_c15_mean"]
    features["btd_13_16"] = features["cmi_c13_mean"] - features["cmi_c16_mean"]
    features["btd_15_16"] = features["cmi_c15_mean"] - features["cmi_c16_mean"]
    
    return features


def process_goes_file(file_path, ix, iy, window=WINDOW_SIZE):
    """
    Abre un NetCDF individual, extrae el vector de características y concatena 
    el timestamp indexado.
    """
    with xr.open_dataset(file_path) as ds:
        features = extract_oaq_features(ds, ix, iy, window)
        features["timestamp"] = pd.to_datetime(ds.t.values)
        
    return features


def safe_process_goes_file(file_path, ix, iy, date, window=WINDOW_SIZE):
    """
    Encapsula el procesamiento para aislar archivos corruptos en disco 
    sin interrumpir la ejecución del pipeline principal.
    """
    try:
        return process_goes_file(file_path, ix, iy, window)
    except Exception as e:
        log_failed_file(date, file_path.name, e)
        return None

In [12]:
# ==============================================================================
# 8. EJECUCIÓN DE PRUEBAS UNITARIAS (SANITY CHECK)
# ==============================================================================

if sample_file and sample_file.exists():
    # Test 1: Localización del píxel del observatorio
    ix, iy = get_oaq_pixel(sample_file)
    print(f" Coordenadas del Píxel OAQ calculadas: (ix={ix}, iy={iy})")
    
    # Test 2: Simulación de procesamiento seguro
    test_date = "2026-01-24" # Fecha correspondiente a los primeros archivos detectados
    result = safe_process_goes_file(sample_file, ix, iy, date=test_date)
    
    if result is not None:
        print("✓ Archivo procesado con éxito.")
        print(f" Timestamp de la observación: {result['timestamp']}")
        print(f" Métricas extraídas ({len(result)}):")
        
        # Mostramos una vista ordenada de las estadísticas y los BTD
        for key, val in result.items():
            if key != "timestamp":
                print(f"  • {key}: {val:.4f}")
    else:
        print(" La prueba falló. Revisa el archivo 'failed_files.csv' para ver los detalles del error.")
else:
    print(" No se dispone de un archivo válido para ejecutar los tests.")

 Coordenadas del Píxel OAQ calculadas: (ix=5196, iy=2722)
✓ Archivo procesado con éxito.
 Timestamp de la observación: 2026-01-24 00:05:07.031083008
 Métricas extraídas (16):
  • cmi_c13_mean: 265.2290
  • cmi_c13_std: 4.2567
  • cmi_c13_min: 256.9574
  • cmi_c13_max: 275.1476
  • cmi_c15_mean: 260.5743
  • cmi_c15_std: 4.5698
  • cmi_c15_min: 251.8212
  • cmi_c15_max: 271.2380
  • cmi_c16_mean: 250.0283
  • cmi_c16_std: 3.4716
  • cmi_c16_min: 244.0090
  • cmi_c16_max: 257.8344
  • btd_13_15: 4.6548
  • btd_13_16: 15.2008
  • btd_15_16: 10.5460


In [13]:
# ==============================================================================
# 9. LOGÍSTICA DE RECONOCIMIENTO DIARIO Y ORQUESTACIÓN
# ==============================================================================

def generate_date_range(start_date=START_DATE, end_date=END_DATE):
    """
    Genera un listado secuencial de fechas en formato YYYY-MM-DD.
    """
    dates = pd.date_range(start=start_date, end=end_date, freq="D")
    return [d.strftime("%Y-%m-%d") for d in dates]


def get_day_files(date):
    """
    Localiza los archivos NetCDF del GOES mapeando la estructura de carpetas
    organizadas por Año y Día Juliano dentro del repositorio crudo local.
    """
    date_obj = pd.to_datetime(date)
    year = date_obj.year
    julian_day = f"{date_obj.dayofyear:03d}"

    # Construimos la subcarpeta Año/DíaJuliano correspondiente
    day_dir = RAW_GOES_DIR / str(year) / julian_day

    if not day_dir.exists():
        return []

    return sorted(day_dir.rglob("*.nc"))


def process_day(date, files, ix, iy, window=WINDOW_SIZE):
    """
    Procesa secuencialmente todos los archivos detectados para un día,
    genera el DataFrame ordenado y guarda el checkpoint en Parquet.
    """
    if day_already_processed(date):
        print(f"✓ La fecha {date} ya registra un Parquet finalizado. Saltando.")
        return None

    print(f"\n Procesando fecha: {date} | Archivos encontrados: {len(files)}")
    results = []
    files_processed = 0
    files_failed = 0

    # Iteración con barra de progreso
    for file_path in tqdm(files, desc=f"Día {date}"):
        record = safe_process_goes_file(file_path, ix, iy, date, window)
        if record is not None:
            results.append(record)
            files_processed += 1
        else:
            files_failed += 1

    # Si el día no generó registros válidos
    if len(results) == 0:
        log_processing_day(date, len(files), 0, files_failed, "FAILED")
        print(f" Error: No se extrajeron registros válidos para {date}")
        return None

    # Estructuración e indexación temporal
    df_day = pd.DataFrame(results)
    df_day = df_day.sort_values("timestamp").reset_index(drop=True)

    # Persistencia del Checkpoint Diario
    daily_file = DAILY_DIR / f"{date}.parquet"
    df_day.to_parquet(daily_file, index=False)

    # Registro de estado
    status = "OK" if files_failed == 0 else "PARTIAL"
    log_processing_day(date, len(files), files_processed, files_failed, status)

    print(f" Guardado diario: {daily_file.name} (Procesados: {files_processed} | Fallidos: {files_failed})")
    
    return df_day


def process_period(start_date=START_DATE, end_date=END_DATE):
    """
    Ejecuta el bucle maestro sobre todo el rango temporal parametrizado.
    """
    dates = generate_date_range(start_date, end_date)
    print(f" Inicializando procesamiento histórico de {len(dates)} días.")
    
    # Calculamos los índices del píxel una sola vez usando el primer archivo disponible
    sample_ref = None
    for d in dates:
        files = get_day_files(d)
        if len(files) > 0:
            sample_ref = files[0]
            break
            
    if sample_ref is None:
        raise FileNotFoundError(" Imposible iniciar el periodo: No se detectaron NetCDF crudos en ninguna carpeta diaria.")
        
    ix, iy = get_oaq_pixel(sample_ref)
    print(f" Matriz de proyección acoplada para el OAQ en: (ix={ix}, iy={iy})")

    for date in dates:
        if day_already_processed(date):
            continue

        files = get_day_files(date)
        if len(files) == 0:
            log_processing_day(date, 0, 0, 0, "NO_DATA")
            continue

        try:
            process_day(date, files, ix, iy)
        except Exception as e:
            print(f" Falla controlada en fecha {date}: {e}")
            log_processing_day(date, len(files), 0, len(files), "FAILED")

In [14]:
# ==============================================================================
# 10. ENSAMBLAJE DE DATASET MAESTRO (CONSOLIDACIÓN)
# ==============================================================================

def build_final_dataset():
    """
    Efectúa la lectura e integración vertical de todos los archivos parquet 
    diarios resguardados, removiendo duplicaciones por solapamiento temporal.
    """
    daily_files = sorted(DAILY_DIR.glob("*.parquet"))

    if len(daily_files) == 0:
        print(" No existen archivos de checkpoints diarios (.parquet) en el directorio.")
        return None

    print(f" Unificando {len(daily_files)} parquets diarios en la base maestra...")
    dfs = []

    for file in tqdm(daily_files, desc="Consolidando"):
        try:
            df = pd.read_parquet(file)
            dfs.append(df)
        except Exception as e:
            print(f" Error al abrir checkpoint {file.name}: {e}")

    if len(dfs) == 0:
        print(" No se logró recuperar información de ningún archivo parquet.")
        return None

    # Concatenación vertical masiva
    df_final = pd.concat(dfs, ignore_index=True)

    # Aseguramos el estricto orden cronológico e inmunidad ante registros repetidos
    df_final = df_final.sort_values("timestamp")
    df_final = df_final.drop_duplicates(subset="timestamp")
    df_final = df_final.reset_index(drop=True)

    # Persistencia definitiva del Dataset de Entrenamiento
    df_final.to_parquet(FINAL_PARQUET, index=False)

    print("\n" + "="*75)
    print(" PROCESAMIENTO MESTRO FINALIZADO CON ÉXITO")
    print(f"  • Ruta del dataset maestro: {FINAL_PARQUET}")
    print(f"  • Total de registros válidos para entrenamiento: {len(df_final):,}")
    print(f"  • Total de columnas multiespectrales calculadas: {len(df_final.columns)}")
    print("="*75)

    return df_final

In [15]:
# ==============================================================================
# CELDA DE CONTROL: SIMULACIÓN DE EXTREMO A EXTREMO (CHECKPOINT DE SEGURIDAD)
# ==============================================================================

print("🔍 Iniciando checkpoint de seguridad del pipeline...")

# 1. Buscar el primer día que realmente contenga archivos NetCDF para la prueba
fechas_rango = generate_date_range(START_DATE, END_DATE)
dia_prueba = None
archivos_prueba = []

for fecha in fechas_rango:
    archivos = get_day_files(fecha)
    if len(archivos) > 0:
        dia_prueba = fecha
        archivos_prueba = archivos[:3]  # Tomamos solo 3 archivos para una prueba rápida
        break

if not dia_prueba:
    print("❌ ERROR: No se encontró ningún directorio con archivos NetCDF crudos en el rango configurado.")
    print(f"   Verifica la ruta: {RAW_GOES_DIR}")
else:
    print(f"✓ Día detectado para la prueba: {dia_prueba} ({len(archivos_prueba)} archivos seleccionados para simulación)")
    
    try:
        # 2. Validar cálculo del píxel del OAQ
        ix_test, iy_test = get_oaq_pixel(archivos_prueba[0])
        print(f"✓ Coordenadas del píxel OAQ calculadas con éxito: (ix={ix_test}, iy={iy_test})")
        
        # 3. Procesar los archivos de prueba usando una fecha ficticia de test
        fecha_virtual = "2026-99-99_TEST"
        path_test_parquet = DAILY_DIR / f"{fecha_virtual}.parquet"
        
        # Forzamos una limpieza previa por si quedó algún rastro anterior
        if path_test_parquet.exists():
            path_test_parquet.unlink()
            
        print("🚀 Ejecutando simulación de procesamiento diario (`process_day`)...")
        df_test_dia = process_day(date=fecha_virtual, files=archivos_prueba, ix=ix_test, iy=iy_test)
        
        # 4. Validar la estructura del DataFrame generado
        if df_test_dia is not None and path_test_parquet.exists():
            print(f"✓ Checkpoint diario generado: {path_test_parquet.name} | Shape: {df_test_dia.shape}")
            
            # 5. Simular la consolidación final (`build_final_dataset`) solo con este archivo virtual
            print("📦 Probando lógica de consolidación masiva...")
            dfs_consolidacion = [pd.read_parquet(path_test_parquet)]
            df_final_virtual = pd.concat(dfs_consolidacion, ignore_index=True)
            df_final_virtual = df_final_virtual.drop_duplicates(subset="timestamp").sort_values("timestamp")
            
            print(f"✓ Lógica de integración completada. Columnas resultantes: {list(df_final_virtual.columns)}")
            
            # Limpieza del entorno de pruebas
            path_test_parquet.unlink()
            print("🧹 Archivos temporales de prueba removidos del disco.")
            print("\n" + "="*75)
            print("✅ ¡DIAGNÓSTICO EXITOSO! Todas las funciones respondieron correctamente sin errores.")
            print("   El pipeline está 100% listo para ejecutarse de forma masiva en producción.")
            print("="*75)
        else:
            print("❌ ERROR: El proceso terminó pero no se generó el archivo Parquet temporal de prueba.")
            
    except Exception as e:
        print(f"\n💥 CRITICAL ERROR DETECTADO DURANTE EL DIAGNÓSTICO:")
        print(f"   Tipo de error: {type(e).__name__}")
        print(f"   Mensaje detallado: {str(e)}")
        print("\n⚠️ Revisa la sintaxis de la celda de funciones antes de continuar.")

🔍 Iniciando checkpoint de seguridad del pipeline...
✓ Día detectado para la prueba: 2026-01-24 (3 archivos seleccionados para simulación)
✓ Coordenadas del píxel OAQ calculadas con éxito: (ix=5196, iy=2722)
🚀 Ejecutando simulación de procesamiento diario (`process_day`)...

 Procesando fecha: 2026-99-99_TEST | Archivos encontrados: 3


Día 2026-99-99_TEST: 100%|██████████| 3/3 [00:00<00:00, 18.63it/s]


 Guardado diario: 2026-99-99_TEST.parquet (Procesados: 3 | Fallidos: 0)
✓ Checkpoint diario generado: 2026-99-99_TEST.parquet | Shape: (3, 16)
📦 Probando lógica de consolidación masiva...
✓ Lógica de integración completada. Columnas resultantes: ['cmi_c13_mean', 'cmi_c13_std', 'cmi_c13_min', 'cmi_c13_max', 'cmi_c15_mean', 'cmi_c15_std', 'cmi_c15_min', 'cmi_c15_max', 'cmi_c16_mean', 'cmi_c16_std', 'cmi_c16_min', 'cmi_c16_max', 'btd_13_15', 'btd_13_16', 'btd_15_16', 'timestamp']
🧹 Archivos temporales de prueba removidos del disco.

✅ ¡DIAGNÓSTICO EXITOSO! Todas las funciones respondieron correctamente sin errores.
   El pipeline está 100% listo para ejecutarse de forma masiva en producción.


In [20]:
# ==============================================================================
# 12. MOTOR DE DESCARGA (GOES2GO) Y MONITOREO DE ESPACIO EN DISCO
# ==============================================================================

# Inicialización global única de la interfaz de descarga de goes2go
goes_downloader = GOES(satellite="noaa-goes18", domain="F")


def download_goes_hour(start_time, end_time):
    """
    Descarga los NetCDF del GOES para un intervalo horario específico y registra 
    métricas de rendimiento de red.
    """
    print(f" Solicitando fragmento satelital: {start_time:%H:%M} -> {end_time:%H:%M}")
    t0 = time.time()
    
    # Descarga directa vía API
    files_df = goes_downloader.timerange(start=start_time, end=end_time)
    
    elapsed = time.time() - t0
    print(f"   • Archivos descargados: {len(files_df)} | Tiempo de red: {elapsed:.1f} s")
    return files_df


def download_goes_day(date, total_hours=24):
    """
    Descarga secuencialmente bloques horarios de datos GOES para un día específico
    con el fin de evitar saturación de ancho de banda o caídas de conexión.
    """
    day_t0 = time.time()
    print("\n" + "=" * 60)
    print(f" INICIANDO DESCARGA HISTÓRICA GOES - FECHA: {date}")
    print("=" * 60)

    day_start = pd.to_datetime(date)
    all_files = []

    for hour in range(total_hours):
        start_time = day_start + timedelta(hours=hour)
        end_time = start_time + timedelta(hours=1)
        print(f"\n[Bloque {hour+1:02d}/{total_hours}]")
        
        try:
            files_df = download_goes_hour(start_time=start_time, end_time=end_time)
            all_files.append(files_df)
        except Exception as e:
            print(f" Falla en descarga horaria {hour:02d}: {e}")

    if len(all_files) == 0:
        print(f" Descarga fallida: Cero archivos recuperados para el día {date}")
        return pd.DataFrame()

    final_df = pd.concat(all_files, ignore_index=True)
    elapsed = time.time() - day_t0
    print("\n" + "=" * 60)
    print(f"✓ Resumen de descarga diaria finalizada con éxito.")
    print(f"   • Total archivos listados: {len(final_df)} | Tiempo total: {elapsed/60:.1f} minutos")
    print("=" * 60)

    return final_df


def cleanup_day_files(date):
    """
    Remueve físicamente los archivos NetCDF pesados y sus respectivas subcarpetas 
    de Día Juliano una vez validadas las firmas de los Parquets.
    """
    dt = datetime.strptime(date, "%Y-%m-%d")
    year = dt.strftime("%Y")
    doy = dt.strftime("%j")

    # Mapeo exacto de la estructura juliana generada por goes2go en RAW_GOES_DIR
    day_dir = RAW_GOES_DIR / year / doy

    if not day_dir.exists():
        print(f" Omitiendo limpieza: No se detectó la carpeta juliana {day_dir}")
        return

    deleted_files = 0
    for nc_file in day_dir.rglob("*.nc"):
        try:
            nc_file.unlink()
            deleted_files += 1
        except Exception as e:
            print(f" No se pudo eliminar el archivo crudo {nc_file.name}: {e}")

    # Eliminación de la raíz del día para no dejar directorios vacíos en Windows
    try:
        shutil.rmtree(day_dir)
    except Exception as e:
        print(f" No se pudo remover el contenedor vacío {day_dir.name}: {e}")
        return

    print("=" * 60)
    print(f"  LIMPIEZA POST-PROCESAMIENTO COMPLETADA ({date})")
    print(f"   • NetCDF crudos purgados del disco: {deleted_files}")
    print(f"   • Carpeta juliana eliminada de raíz: {day_dir.name}")
    print("=" * 60)

In [22]:
# ==============================================================================
# 13. EJECUCIÓN MIGRATORIA INTERNA Y REPORTES DE PRODUCCIÓN
# ==============================================================================

def process_period(start_date=START_DATE, end_date=END_DATE):
    """
    Orquesta de punta a punta la automatización completa:
    Checkpoint -> Descarga Satelital -> Extracción -> Validación Parquet -> Limpieza NetCDF
    """
    period_t0 = time.time()
    dates = generate_date_range(start_date, end_date)

    processed_days = 0
    skipped_days = 0
    failed_days = 0

    print("=" * 75)
    print(" INICIALIZANDO EJECUCIÓN MAESTRA DEL PIPELINE (END-TO-END)")
    print(f"   • Ventana temporal de análisis: {start_date} al {end_date}")
    print(f"   • Total de jornadas a procesar: {len(dates)} días")
    print("=" * 75)

    # 1. Ejecutar una pre-búsqueda rápida para fijar la matriz de proyección espacial (ix, iy)
    sample_ref = None
    for d in dates:
        files = get_day_files(d)
        if len(files) > 0:
            sample_ref = files[0]
            break
            
    # Si no hay archivos locales previos, descargamos temporalmente el primer bloque horario del rango
    if sample_ref is None:
        print(" No se detectaron NetCDF locales de referencia. Descargando muestra inicial...")
        try:
            day_start = pd.to_datetime(start_date)
            sample_df = download_goes_hour(day_start, day_start + timedelta(hours=1))
            files = get_day_files(start_date)
            if len(files) > 0:
                sample_ref = files[0]
        except Exception as e:
            raise RuntimeError(f" Error fatal al intentar calibrar el píxel de referencia: {e}")

    if sample_ref is None:
        raise FileNotFoundError(" Imposible arrancar el pipeline: Sin acceso a un NetCDF base para calcular ix, iy.")

    ix, iy = get_oaq_pixel(sample_ref)
    print(f" Calibración completada para el OAQ: Píxel fijo en (ix={ix}, iy={iy})")

    # 2. Bucle principal sobre el rango cronológico
    for i, date in enumerate(dates):
        print("\n" + "=" * 75)
        print(f" JORNADA [{i+1}/{len(dates)}] — FECHA UNIFICADA: {date}")
        print("=" * 75)

        # A. Checkpoint previo para no re-descargar ni re-procesar datos existentes
        if day_already_processed(date):
            print(f"    Checkpoint detectado. Los datos de {date} ya están consolidados. Saltando.")
            skipped_days += 1
            continue

        try:
            # B. Ejecutar la descarga automatizada del día completo
            files_downloaded = download_goes_day(date)
            if len(files_downloaded) == 0:
                failed_days += 1
                continue

            # C. Indexar los archivos descargados localmente en su ruta juliana
            files = get_day_files(date)
            if len(files) == 0:
                print(f"    Error crítico: La descarga finalizó pero no se localizaron NetCDF en disco.")
                failed_days += 1
                continue

            # D. Procesar los archivos del día, extraer métricas y salvar parquet diario
            process_day(date=date, files=files, ix=ix, iy=iy)

            # E. Auditoría estricta de salida del parquet diario generado
            parquet_file = DAILY_DIR / f"{date}.parquet"
            if not parquet_file.exists():
                print(f"    Error en consistencia: El archivo {parquet_file.name} no fue creado.")
                failed_days += 1
                continue

            # F. Validación estructural de datos antes de limpiar los NetCDF originales
            df_audit = pd.read_parquet(parquet_file)
            if len(df_audit) == 0:
                print(f"    Alerta de integridad: El parquet diario de {date} está vacío.")
                print("      Se suspende la limpieza física de los NetCDF para revisión forense.")
                failed_days += 1
                continue

            # G. Si todo está perfecto, procedemos a liberar espacio en disco duro
            cleanup_day_files(date)
            processed_days += 1
            print(f"    Procesamiento completo e integridad validada para el día: {date}")

        except Exception as e:
            print(f"    Falla inesperada durante la ejecución automatizada de {date}: {e}")
            failed_days += 1
            continue

    # 3. Consolidación e integración vertical definitiva de la matriz maestra
    print("\n" + "=" * 75)
    print(" INICIANDO ENSAMBLAJE DE LA MATRIZ DE ENTRENAMIENTO FINAL")
    print("=" * 75)

    dataset_ok = False
    try:
        build_final_dataset()
        dataset_ok = True
    except Exception as e:
        print(f" Falló la consolidación del dataset final: {e}")

    elapsed = time.time() - period_t0
    print("\n" + "=" * 75)
    print(" REPORTE DE CIERRE GENERAL DEL PIPELINE DE PRODUCCIÓN")
    print("=" * 75)
    print(f"   • Días procesados con éxito  : {processed_days}")
    print(f"   • Días omitidos (Checkpoints): {skipped_days}")
    print(f"   • Días con fallas registradas : {failed_days}")
    print(f"   • Tiempo total de ejecución   : {elapsed/60:.1f} minutos")
    print(f"   • Estado final de la matriz   : {'✓ OPERATIVA / GENERADA' if dataset_ok else '✗ CORRUPTA / NO GENERADA'}")
    print("=" * 75)

In [ ]:
# ==============================================================================
# 14. DISPARADOR DE PRODUCCIÓN (PRIMERA SEMANA DE ENERO)
# ==============================================================================

# 1. Ejecutar el pipeline secuencial (Descarga -> Extracción -> Limpieza)
process_period(start_date=START_DATE, end_date=END_DATE)

# 2. Consolidar el dataset maestro final para los modelos LSTM/GNN
df_final = build_final_dataset()

In [ ]:
# ==============================================================================
# 15. INFORME DE DIAGNÓSTICO DEL REPOSITORIO DE DATOS
# ==============================================================================

def print_repository_summary():
    """
    Imprime un informe en consola detallando el estado actual de los archivos resguardados.
    """
    daily_parquets = list(DAILY_DIR.glob("*.parquet"))
    
    print("=" * 60)
    print("       SISTEMA ASTROFORECAST - RESUMEN DE COMPONENTES")
    print("=" * 60)
    print(f"📁 Directorio de crudos (NetCDF)    : {RAW_GOES_DIR}")
    print(f"📁 Checkpoints diarios (.parquet)   : {len(daily_parquets)} archivos")
    print(f"📝 Registro de Logs de Proceso      : {'✓ Presente' if PROCESSING_LOG.exists() else '✗ Ausente'}")
    print(f"⚠️ Registro de Archivos Corruptos   : {'✓ Presente' if FAILED_FILES_LOG.exists() else '✗ Ausente'}")
    print("-" * 60)
    
    if FINAL_PARQUET.exists():
        size_mb = FINAL_PARQUET.stat().st_size / (1024 * 1024)
        print(f"🏁 Dataset Final Maestro (Parquet)  : ✓ OPERATIVO ({size_mb:.2f} MB)")
        print(f"   • Ubicación: {FINAL_PARQUET.name}")
    else:
        print("🏁 Dataset Final Maestro (Parquet)  : ✗ NO DETECTADO / PENDIENTE")
    print("=" * 60)

# Lanzamiento automático del estado inicial
print_repository_summary()

       SISTEMA ASTROFORECAST - RESUMEN DE COMPONENTES
📁 Directorio de crudos (NetCDF)    : C:\Users\Brian OAQ\data\noaa-goes18\ABI-L2-MCMIPF
📁 Checkpoints diarios (.parquet)   : 2 archivos
📝 Registro de Logs de Proceso      : ✓ Presente
⚠️ Registro de Archivos Corruptos   : ✓ Presente
------------------------------------------------------------
🏁 Dataset Final Maestro (Parquet)  : ✓ OPERATIVO (0.03 MB)
   • Ubicación: goes_oaq_dataset.parquet


# AstroForecast — Pipeline de Procesamiento Espacial GOES-18 (Fase 4)

Este repositorio contiene el módulo de ingeniería de datos y procesamiento espacial automatizado para el proyecto **AstroForecast**. Su objetivo es descargar, calibrar, extraer y consolidar variables físicas multiespectrales provenientes del satélite **GOES-18** (producto `ABI-L2-MCMIPF` - Disco Completo) apuntando específicamente a las coordenadas de la **Unidad de Clima Espacial del Observatorio Astronomico de Quito (OAQ)**.

La matriz resultante está optimizada para alimentar modelos de aprendizaje profundo (*Deep Learning*) secuenciales y espaciotemporales (tales como **LSTM, GNN y ST-GCN**) dedicados a la predicción de nubosidad local para optimizar las observaciones astronómicas.

---

## Arquitectura del Pipeline y Flujo de Datos

El procesamiento de imágenes satelitales de disco completo presenta un desafío crítico: cada archivo NetCDF bruto (.nc) pesa entre 200 y 300 MB. Procesar un rango histórico continuo de varios meses requeriría terabytes de almacenamiento local libre. 

Para resolver esto, el pipeline de la **Fase 4** procesa los datos en **lotes diarios cerrados con eliminación inmediata de crudos**, manteniendo el requisito de almacenamiento constante en un búfer temporal máximo de ~25 GB.

### Flujo de Ejecución Secuencial (Día a Día)

                 Rango de Fechas (START_DATE -> END_DATE)
                                    │
                                    ▼
                   process_period(start_date, end_date)
                                    │
                                    ▼
                      Para cada día del período...
                                    │
              ┌─────────────────────┴─────────────────────┐
              │ 1. Checkpoint: ¿Día ya procesado?         │──► [SÍ] ──► Saltar fecha
              └─────────────────────┬─────────────────────┘
                                    │ [NO]
                                    ▼
              ┌───────────────────────────────────────────┐
              │ 2. Descarga Horaria Controlada (goes2go)  │──► 144 archivos NetCDF / día
              └─────────────────────┬─────────────────────┘
                                    │
                                    ▼
              ┌───────────────────────────────────────────┐
              │ 3. Calibración e Indexación Espacial      │──► Ventana 5x5 sobre píxel OAQ
              └─────────────────────┬─────────────────────┘
                                    │
                                    ▼
              ┌───────────────────────────────────────────┐
              │ 4. Guardado y Auditoría del Checkpoint    │──► Guardar YYYY-MM-DD.parquet
              └─────────────────────┬─────────────────────┘
                                    │
                                    ▼
              ┌───────────────────────────────────────────┐
              │ 5. Liberación Física de Disco (shutil)    │──► Eliminación de NetCDF brutos
              └─────────────────────┬─────────────────────┘
                                    │
                                    ▼
                        ¿Última fecha alcanzada?
                                    │
                                    ▼
              ┌───────────────────────────────────────────┐
              │ 6. Ensamblaje del Dataset Maestro         │──► goes_oaq_dataset.parquet
              └───────────────────────────────────────────┘

---

## Componentes y Optimización de Ingeniería

### 1. Calibración Geoestacionaria Inversa (pyproj)
Para evitar realizar búsquedas espaciales pesadas de coordenadas geográficas en cada imagen de satélite, la función `get_oaq_pixel` proyecta las coordenadas geodésicas del Observatorio Astronómico de Quito (Lat: -0.2150, Lon: -78.5025) al plano de proyección geoestacionaria del satélite GOES-18 usando sus metadatos de perspectiva espacial fijos (perspective_point_height, longitude_of_projection_origin y sweep_angle_axis). Esto mapea de forma exacta las coordenadas al índice de matriz de píxeles (ix, iy) una sola vez al arrancar el proceso, acelerando la extracción espacial a más de 13 imágenes por segundo.

### 2. Extracción Espectral Focalizada
Para caracterizar la nubosidad (densidad, tipo y altura de nubes) sobre el OAQ, el script extrae estadísticas descriptivas en una ventana espacial de 5x5 píxeles centrada en el observatorio para los tres canales infrarrojos críticos del instrumento ABI:
* **CMI_C13 (Infrarrojo de onda larga - 10.3 µm):** Utilizado para estimar la temperatura de brillo en la cima de las nubes y mapear la estructura general de la nubosidad.
* **CMI_C15 (Infrarrojo de sucia ventana de agua - 12.3 µm):** Sensible al vapor de agua en niveles bajos y a la emisividad de nubes delgadas.
* **CMI_C16 (Infrarrojo de absorción de CO2 - 13.3 µm):** Utilizado para estimar la altura de las nubes debido a la fuerte absorción de dióxido de carbono.

Para cada canal dentro de la ventana de recorte se calculan de manera vectorizada la Media, Desviación Estándar, Mínimo y Máximo.

### 3. Diferencias de Temperatura de Brillo (BTD)
Las diferencias térmicas entre bandas son predictores físicos de alta fidelidad para discernir nubes delgadas de hielo (cirros) de nubes bajas y niebla. El pipeline calcula de forma nativa:
* **BTD 13 - 15:** Diferencia espectral clásica para identificar nubes cirrus y espesor óptico.
* **BTD 13 - 16:** Permite estimar la altitud del tope nuboso y fases de agua/hielo.
* **BTD 15 - 16:** Útil para la discriminación microfísica de las partículas de nube.

### 4. Tolerancia a Fallos y Registro de Logs de Auditoría
* **Aislamiento de NetCDF Corruptos (safe_process_goes_file):** Si un archivo se descargó de manera incompleta o su estructura de compresión HDF5 está corrupta, el script atrapa la excepción, registra el nombre del archivo y el error exacto en `logs/failed_files.csv` y continúa procesando el lote diario sin detener el pipeline.
* **Integridad del Checkpoint:** Antes de borrar físicamente los archivos NetCDF pesados del disco local, el script carga el .parquet diario generado y audita que tenga filas válidas registradas. Si el archivo está vacío o corrupto, detiene la limpieza para permitir una revisión manual del operador.

---

## Estructura del Dataset de Salida (goes_oaq_dataset.parquet)

El archivo final consolidado contiene 16 variables estructuradas en formato tabular de alta compresión:

* `timestamp`: Marca de tiempo central de la observación en formato UTC (datetime64[ns]).
* `cmi_c13_mean`, `cmi_c13_std`, `cmi_c13_min`, `cmi_c13_max`: Estadísticos del canal 13.
* `cmi_c15_mean`, `cmi_c15_std`, `cmi_c15_min`, `cmi_c15_max`: Estadísticos del canal 15.
* `cmi_c16_mean`, `cmi_c16_std`, `cmi_c16_min`, `cmi_c16_max`: Estadísticos del canal 16.
* `btd_13_15`: Diferencia de temperatura de brillo Canal 13 - Canal 15.
* `btd_13_16`: Diferencia de temperatura de brillo Canal 13 - Canal 16.
* `btd_15_16`: Diferencia de temperatura de brillo Canal 15 - Canal 16.

---

## Configuración y Requisitos de Software

Este módulo se ejecuta dentro del entorno virtual de conda `sat_env`.

### Dependencias Principales
* **xarray & h5netcdf:** Manejo eficiente y de lectura perezosa para archivos de formato jerárquico de datos.
* **pyproj:** Proyecciones y transformaciones cartográficas avanzadas.
* **goes2go:** Descarga automática directa desde los repositorios S3 de la NOAA/AWS.
* **pandas & pyarrow:** Estructuración de datos y persistencia en archivos optimizados Parquet.
* **tqdm:** Barras de progreso interactivas para monitoreo en consola.

### Estructura de Rutas del Proyecto
```text
OAQ-AstroForecast/
│
├── data/
│   ├── raw/
│   │   └── goes/                 <-- Almacenamiento temporal de NetCDF crudos (Año/DíaJuliano)
│   │
│   └── processed/
│       └── goes/
│           ├── daily/            <-- Checkpoints Parquet de cada día (YYYY-MM-DD.parquet)
│           ├── logs/             <-- processing_log.csv y failed_files.csv
│           └── goes_oaq_dataset.parquet  <-- DATASET FINAL CONSOLIDADO